In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv() # carrega a API da I.A do .env

model = init_chat_model("openai:gpt-4o-mini")

In [ ]:
from langchain.tools import tool

@tool
def buscar_clima(cidade: str) -> str:
    """Retorna o clima atual de uma cidade informada
    
    Args:
        cidade: nome da cidade que se quer consultar.
    """
    # Aqui é um exemplo "fake": em um app real você chamaria uma API de clima.
    return f"Em {cidade} está ensolarado, 25°C."    

In [ ]:
from langchain.tools import tool

@tool
def calcular(expressao: str) -> str:
    """Calcula o resultado de uma expressão matemática simples.
    
    Args:
        expressao: expressão em texto, ex: "12 * (3 + 4)".
    """
    # eval é usado aqui só para fins didáticos - NÃO use eval com entrada
    # não confiável em produção (risco de segurança).
    resultado = eval(expressao)
    return f"O resultado de {expressao} é {resultado}."

@tool
def buscar_clima(cidade: str) -> str:
    """Retorna o clima atual de uma cidade informada.
    
    Args:
        cidade: nome da cidade que se quer consultar.
    """
    # Dados fake só para o exemplo (sem API externa)
    base = {"São Paulo": "nublado, 19°C", "Recife": "ensolarado, 30°C"}
    return f"Clima em {cidade}: {base.get(cidade, 'ensolarado, 25°C')}."

In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent

load_dotenv()

agent = create_agent(
    model="openai:gpt-4o-mini",      #string "provedor:modelo" (ou modelo instanciado)
    tools=[calcular, buscar_clima],             # as ferramentas que agent pode usar
    system_prompt="Você é um assistente útil e responde em português do Brasil.",
)

In [ ]:
res = agent.invoke({
    "messages": [
        {"role": "user", "content": "Quanto é 12 * (3 + 4)? E como está o clima em Recife?"}
    ]
})

# A resposta final é a ÚLTIMA mensagem do estado
print(res["messages"][-1].content)
# -> "12 * (3 + 4) é 84. Em Recife está ensolarado, 30°C."

In [ ]:
res = agent.invoke({
    "messages": [
        {"role": "user", "content": "Qual o clima em BKK?"}
    ]
})

In [ ]:
print(res["messages"][-1].content)

In [ ]:
for m in res["messages"]:
    m.pretty_print()

In [ ]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "Como está o clima em São Paulo?"}]},
    stream_mode="values",
):
    # mostra a última mensagem de cada etapa (chamada de tool, resultado, resposta...)
    chunk["messages"][-1].pretty_print()